# От качества исходной таблицы к надежности анализа

Представим ситуацию, в которой инженер получил табличный набор данных и собирается строить классификационную модель устойчивости режима. Первый риск здесь связан не с алгоритмом, а с тем, насколько сама таблица пригодна для дальнейшего анализа.

<div style="border-left: 4px solid #1f4b99; background: #eef5ff; padding: 12px 14px; margin: 12px 0;">
<strong>Учебная задача.</strong><br/>
Необходимо не просто открыть файл, а установить, насколько он структурирован, есть ли в нем пропуски и дубликаты, как устроены признаки и можно ли переходить к моделированию без дополнительной очистки.
</div>

## Исходные данные

В блокноте используется тематический набор <em>Electrical Grid Stability Simulated Data</em>. Хотя этот набор сравнительно чистый, именно поэтому он удобен как первый пример: на нем можно увидеть, какие характеристики качества следует проверять всегда, даже если таблица выглядит аккуратно уже при первом открытии.

## ????? ????????? ????? ????????

????? ?????????? ???????? ?????? ???? ???????:

- ??? ??????? ???????? ? ????????? ?????? ??????;
- ????? ???????? ???????? ????????, ? ????? ? ????????;
- ??????? ?? ???????? ??????? ? ????????????? ???????;
- ????? ??????? ?????? ???????????? ????? ? ???????? `CSV`, `XLSX`, `TXT` ? `JSON`;
- ????? ?? ???????????? ??????? ??? ???????? ????? ??? ??????? ?????? ?????????????.


## ?????? ????????

### 1. ???????? ?????????? ???????? ?????
???????????? ????? ?????? ?? `data/sample/`, ????? ??????? ?????????? ???????? ????? ???????? ???????????? ???????????.

### 2. ????????? ???????? ?????????
??????????? ???????????, ???????? ????????? ? ?????? ??????.

### 3. ??????? ????????
??????????? ???? ??????, ????????, ????????? ? ????????????? ???????.

### 4. ?????? ? ??????? ????????? ??????
?? ???????? ???????????? ?????????? ???????????? ?????? `CSV`, `XLSX`, `TXT` ? `JSON` ?????? ??????? ???????? ???????????.

### 5. ?????
????????????? ?????????? ? ?????????? ?????? ?????? ? ??????????? ??????? ? ? ???, ??? ?????? ????????? ?????? ?? ?????? ????????.


In [ ]:
import pandas as pd
from IPython.display import HTML, display

from src.display_utils import display_callout, display_metric_table, display_markdown_list
from src.project_paths import sample_data_dir

DATA_PATH = sample_data_dir() / "classification" / "electrical_grid_stability.csv"
FORMAT_DIR = sample_data_dir() / "formats"

df = pd.read_csv(DATA_PATH)
df.head()


## Что видно уже на первом шаге

Таблица содержит набор числовых признаков и целевые поля, относящиеся к устойчивости режима. Уже на этом этапе полезно различать входные параметры и итоговую метку класса, чтобы далее не допустить утечки целевой информации в признаки.

In [ ]:
overview = pd.DataFrame(
    {
        "rows": [len(df)],
        "columns": [df.shape[1]],
        "numeric_columns": [df.select_dtypes(include="number").shape[1]],
        "object_columns": [df.select_dtypes(include="object").shape[1]],
    }
)
overview

## Интерпретация структуры

Набор почти полностью состоит из числовых признаков. Это облегчает переход к моделированию, но не отменяет обязательной проверки пропусков, дубликатов и классового баланса.

In [ ]:
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing_share": df.isna().mean().round(4),
        "unique_values": df.nunique(dropna=False),
    }
)
profile

## Интерпретация качества

Профиль показывает, что набор хорошо структурирован: типы читаются корректно, пропуски отсутствуют, а число уникальных значений у целевой метки ограничено двумя классами. Такой результат не означает, что этап контроля качества можно пропускать; напротив, он показывает, каким должен быть исходный набор данных перед моделированием.

In [ ]:
duplicate_count = int(df.duplicated().sum())
class_distribution = df["stabf"].value_counts().rename_axis("class").reset_index(name="count")

metrics = {
    "duplicate_rows": float(duplicate_count),
    "max_missing_share": float(df.isna().mean().max()),
    "stable_share": float((df["stabf"] == "stable").mean()),
    "unstable_share": float((df["stabf"] == "unstable").mean()),
}

display_metric_table(metrics, decimals=4)
class_distribution

## Почему важно проверить баланс классов

Даже на чистом наборе данных распределение целевой метки имеет методическое значение: если один класс сильно преобладает, обычная accuracy перестает быть достаточной метрикой. Здесь классы распределены сравнительно ровно, поэтому задача удобна для первого знакомства с классификацией.

In [ ]:
issues = []
if duplicate_count:
    issues.append("обнаружены дублирующиеся строки")
if df.isna().any().any():
    issues.append("обнаружены пропуски в данных")
if not issues:
    issues.append("критических проблем качества на первичном уровне не выявлено")

display_markdown_list("Итог первичной проверки", issues)
display_callout(
    "Вывод по набору данных",
    "Набор Electrical Grid Stability Simulated Data пригоден для учебной задачи классификации и может использоваться как базовый тематический пример для дальнейшего анализа.",
    tone="success",
)

## ??? ???? ? ??? ?? ??????? ?????? ???????? ? ??????? ????????? ??????

? ???????? ???????? ????????????? ????? ??????? ?? ???????? ?????? ? ????? ????????? ???????. ????? ?????????? ???????? ? ???? `CSV`, ????? ? ??? ??????????? ??????? `XLSX`, ????? ? ??? ????????? ??????? `TXT`, ? ????????? ???????? ? ???????? ??????? ??????????? ? `JSON`.

???? ????????, ??? ??? ??????? ???????? ?? ???????? ???????????? ?????????? ?? ???????????. ????????????? ??? ?????? ??????, ?? ??????????? ?????? ????? ???, ??? ??? ????? ??????????? ????? ????????????? ???? ? ????? ?????????? ?????????? ??? ??????????? ???????.


In [ ]:
csv_fragment = pd.read_csv(FORMAT_DIR / "grid_stability_fragment.csv")
xlsx_fragment = pd.read_excel(FORMAT_DIR / "combined_cycle_power_plant_fragment.xlsx")
txt_fragment = pd.read_csv(FORMAT_DIR / "household_power_fragment.txt", sep=";")
catalog = pd.read_json(FORMAT_DIR / "dataset_catalog.json")

format_overview = pd.DataFrame(
    [
        {"file": "grid_stability_fragment.csv", "format": "CSV", "rows": len(csv_fragment), "columns": csv_fragment.shape[1], "loading_function": "pd.read_csv(...)"},
        {"file": "combined_cycle_power_plant_fragment.xlsx", "format": "XLSX", "rows": len(xlsx_fragment), "columns": xlsx_fragment.shape[1], "loading_function": "pd.read_excel(...)"},
        {"file": "household_power_fragment.txt", "format": "TXT", "rows": len(txt_fragment), "columns": txt_fragment.shape[1], "loading_function": "pd.read_csv(..., sep=";")"},
        {"file": "dataset_catalog.json", "format": "JSON", "rows": len(catalog), "columns": catalog.shape[1], "loading_function": "pd.read_json(...)"},
    ]
)

display(format_overview)

preview_frames = [
    ("CSV: ???????? ?????? ???????????? ????", csv_fragment.head(3)),
    ("XLSX: ???????? ?????? ?? ???????? ??????????????", xlsx_fragment.head(3)),
    ("TXT: ???????? ?????????? ?????????? ????", txt_fragment.head(3)),
    ("JSON: ??????? ????????? ??????? ??????", catalog),
]

for title, frame in preview_frames:
    display(HTML(f"<h4 style="margin-top: 18px; color: #1f4b99;">{title}</h4>"))
    display(frame)


## ????????????? ???????? ??????

?????? ??????????, ??? ???????? ??????? ?? ???????? ? ???????? ????????. ??? `CSV` ? `TXT` ????? ?????????????? ??????????? ? ????????? ??????????? ?????????, ??? `XLSX` ????? ????????, ??? ?????? ????? ?????????? ?? ?????????? ????? ? ????????? ????????? ??????, ? `JSON` ??????? ?????? ?? ???? ?????????, ? ?????????? ? ??????, ????????? ? ????????????? ??????.

? ???????????? ????? ?????? ??????? ??????, ??? ??? ??? ????? ????? ?????????? ? ????? ??????????? ? ??????????? ???????????? ????? ????????????? ????. ??? ?????? ??????? ???????? ???????????????: ????? ???????????? ??????????? ??????????? ???????? ?? ? ???????????? ???????? ?? ??????? ?????????, ? ? ????????? ??????? ????????, ????????? ??????? ? ???????.


## Итоговый вывод

Первичный контроль качества показал, что тематический набор устойчивости сети хорошо подходит для учебного сценария. Это позволяет использовать его не только в главе о качестве данных, но и далее как опорный пример для классификационного кейса.

## Вопросы для самостоятельной работы

1. Какие дополнительные проверки качества были бы необходимы, если бы набор данных содержал временные метки?
2. Почему целесообразно отделять `stab` и `stabf` при последующем моделировании?
3. В каком случае даже чистый набор данных может приводить к некорректным выводам?

## Источники

1. [Паспорт датасетов](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/datasets.md)
2. [Глава 2 пособия](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/chapters/02-data-sources-and-quality.md)